In [20]:
pip install awscli

Note: you may need to restart the kernel to use updated packages.


In [21]:
{
  "Effect": "Allow",
  "Action": [
    "textract:AnalyzeDocument",
    "s3:GetObject"
  ],
  "Resource": "*"
}

{'Effect': 'Allow',
 'Action': ['textract:AnalyzeDocument', 's3:GetObject'],
 'Resource': '*'}

In [22]:
!pip install boto3

In [24]:
import boto3

session = boto3.Session(profile_name="aiengineer")
textract = session.client("textract")

In [59]:
QUERIES = [
    {
        "Text": "What is the 17-character VIN (vehicle identification number)?",
        "Alias": "vin"
    },
    {
        "Text": "What is the full name of the customer, buyer, purchaser, lessee(s) name or lessee ?",
        "Alias": "customer_name"
    },
    {
        "Text": "What is the sale date or purchase date or effective date of lease?",
        "Alias": "sale_date"
    },
    {
        "Text": "What is the total amount paid or total sale price or Leased vehicle amount?",
        "Alias": "total_amount"
    }
]

In [34]:
bucket = "s3-textract-demo-kavi"
key = "03.pdf"

In [35]:
s3 = session.client("s3")

s3.head_object(Bucket=bucket, Key=key)

{'ResponseMetadata': {'RequestId': 'PKA0EJW4W5N1N5D4',
  'HostId': 'z+9sV9re4IfdLNUSKZU1h3f9pJjOrQ4B3MSodVBLx2TyiaxOp4C96/XMlKYs91eq0Gk5L6/7vFI=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'z+9sV9re4IfdLNUSKZU1h3f9pJjOrQ4B3MSodVBLx2TyiaxOp4C96/XMlKYs91eq0Gk5L6/7vFI=',
   'x-amz-request-id': 'PKA0EJW4W5N1N5D4',
   'date': 'Sun, 03 May 2026 23:23:28 GMT',
   'last-modified': 'Sun, 03 May 2026 03:52:25 GMT',
   'etag': '"e49d58a3fb47257b8f4aff0409a16fc0"',
   'x-amz-server-side-encryption': 'AES256',
   'accept-ranges': 'bytes',
   'content-type': 'application/pdf',
   'content-length': '1082268',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'AcceptRanges': 'bytes',
 'LastModified': datetime.datetime(2026, 5, 3, 3, 52, 25, tzinfo=tzutc()),
 'ContentLength': 1082268,
 'ETag': '"e49d58a3fb47257b8f4aff0409a16fc0"',
 'ContentType': 'application/pdf',
 'ServerSideEncryption': 'AES256',
 'Metadata': {}}

In [45]:
response = textract.analyze_document(
    Document={
        "S3Object": {
            "Bucket": bucket,
            "Name": key
        }
    },
    FeatureTypes=["QUERIES"],
    QueriesConfig={
        "Queries": QUERIES
    }
) 

UnsupportedDocumentException: An error occurred (UnsupportedDocumentException) when calling the AnalyzeDocument operation: Request has unsupported document format

In [36]:
s3.download_file(bucket, key, "test.pdf")

In [37]:
s3.head_object(Bucket=bucket, Key=key)["ContentLength"]

1082268

In [38]:
print(key)

03.pdf


In [39]:
s3.download_file(bucket, key, "debug.pdf")

In [40]:
s3.download_file(bucket, key, "debug.pdf")

In [60]:
response = textract.start_document_analysis(
    DocumentLocation={
        "S3Object": {
            "Bucket": bucket,
            "Name": key
        }
    },
    FeatureTypes=["QUERIES"],
    QueriesConfig={
        "Queries": QUERIES
    }
)

job_id = response["JobId"]
print("Job ID:", job_id)

Job ID: a05af8c13d637aa2e07f9a93ce1f2ac09e32b0dd0deed8f567b4c2a46c6ab2b6


In [61]:
import time

while True:
    result = textract.get_document_analysis(JobId=job_id)
    
    status = result["JobStatus"]
    print("Status:", status)
    
    if status in ["SUCCEEDED", "FAILED"]:
        break
    
    time.sleep(3)

Status: IN_PROGRESS
Status: IN_PROGRESS
Status: IN_PROGRESS
Status: SUCCEEDED


In [62]:
def parse_textract_response(response):
    results = {}
    blocks = {b["Id"]: b for b in response["Blocks"]}

    for block in response["Blocks"]:
        if block["BlockType"] == "QUERY":
            alias = block["Query"].get("Alias")

            value = ""
            confidence = 0

            for rel in block.get("Relationships", []):
                if rel["Type"] == "ANSWER":
                    answer = blocks[rel["Ids"][0]]
                    value = answer.get("Text", "")
                    confidence = answer.get("Confidence", 0)

            results[alias] = {
                "value": value,
                "confidence": round(confidence, 2)
            }

    return results


results = parse_textract_response(result)
print(results)

{'vin': {'value': '1C4RJYC67N8577619', 'confidence': 89.0}, 'total_amount': {'value': '$ 74194.50', 'confidence': 28.0}, 'customer_name': {'value': 'VISTAVIEW MANAGEMENT LTD', 'confidence': 85.0}, 'sale_date': {'value': '09/21/2022', 'confidence': 83.0}}
